In [ ]:
from google.colab import files
import pandas as pd

# -----------------------------
# Upload longsilence txt file
# -----------------------------
print("Upload the longsilence TXT file")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
print("Selected file:", file_name)

# -----------------------------
# Read file
# -----------------------------
data = []

with open(file_name) as f:
    for line in f:
        parts = line.split()

        if len(parts) >= 3:
            time = float(parts[1])
            flag = int(parts[2])

            data.append((time, flag))

# -----------------------------
# Extract silence segments
# -----------------------------
silence_segments = []
start = None

for time, flag in data:

    if flag == -1 and start is None:
        start = time

    elif flag == 1 and start is not None:
        silence_segments.append([start, time])
        start = None

# -----------------------------
# Convert to Excel
# -----------------------------
df = pd.DataFrame(
    silence_segments,
    columns=["Silence Start (sec)", "Silence End (sec)"]
)

excel_file = "long_silence_segments.xlsx"
df.to_excel(excel_file, index=False)

print("Excel file created:", excel_file)

# -----------------------------
# Auto download
# -----------------------------
files.download(excel_file)

Upload the longsilence TXT file


Saving englishog_boundary.txt to englishog_boundary.txt
Selected file: englishog_boundary.txt
Excel file created: long_silence_segments.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import re
import pandas as pd
from google.colab import files

# Ask user to upload TXT file
print("Upload the silence detection TXT file")
uploaded = files.upload()

# Get filename
txt_file = list(uploaded.keys())[0]

silence_data = []
start = None

# Read file
with open(txt_file, "r") as f:
    for line in f:

        start_match = re.search(r"silence_start:\s*([0-9.]+)", line)
        end_match = re.search(r"silence_end:\s*([0-9.]+)", line)

        if start_match:
            start = float(start_match.group(1))

        if end_match and start is not None:
            end = float(end_match.group(1))
            duration = end - start

            silence_data.append({
                "Start Time (s)": start,
                "End Time (s)": end,
                "Duration (s)": duration
            })

# Convert to DataFrame
df = pd.DataFrame(silence_data)

# Save Excel
output_file = "long_silence.xlsx"
df.to_excel(output_file, index=False)

print("Excel created!")

# Download automatically
files.download(output_file)

Upload the silence detection TXT file


Saving englishog_sig (7).txt to englishog_sig (7).txt
Excel created!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
from google.colab import files

print("Upload long silence file")
uploaded = files.upload()

filename = list(uploaded.keys())[0]

# Read the file
data = pd.read_csv(filename, delim_whitespace=True, header=None)

data.columns = ["frame", "time", "marker"]

silence_start = None
silence_regions = []

for i, row in data.iterrows():

    if row["marker"] == 1:
        silence_start = row["time"]

    elif row["marker"] == -1 and silence_start is not None:
        silence_end = row["time"]
        duration = silence_end - silence_start

        silence_regions.append([silence_start, silence_end, duration])
        silence_start = None

df = pd.DataFrame(silence_regions, columns=["Start", "End", "Duration"])

df.to_excel("long_silence.xlsx", index=False)

print("Excel file created!")

files.download("long_silence.xlsx")

Upload long silence file


Saving englishog_boundary.txt to englishog_boundary (1).txt
Excel file created!


/tmp/ipykernel_551/2225351981.py:10: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data = pd.read_csv(filename, delim_whitespace=True, header=None)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
from google.colab import files

print("Upload long silence file")
uploaded = files.upload()

filename = list(uploaded.keys())[0]

# read the file
data = pd.read_csv(filename, sep=r"\s+", header=None)
data.columns = ["frame", "time", "marker"]

silence_regions = []

i = 0
while i < len(data):

    if data.iloc[i]["marker"] == 1:

        start_time = data.iloc[i]["time"]
        j = i + 1
        last_end = None

        while j < len(data) and data.iloc[j]["marker"] == -1:
            last_end = data.iloc[j]["time"]
            j += 1

        if last_end is not None:
            duration = last_end - start_time
            silence_regions.append([start_time, last_end, duration])

        i = j
    else:
        i += 1


df = pd.DataFrame(silence_regions, columns=["Start", "End", "Duration"])

df.to_excel("long_silence.xlsx", index=False)

print("Excel file created!")

files.download("long_silence.xlsx")

Upload long silence file


Saving englishog_boundary (7).txt to englishog_boundary (7).txt
Excel file created!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!apt-get install ffmpeg -y

import pandas as pd
import librosa
from google.colab import files
import subprocess

print("Upload long_silence.xlsx")
uploaded = files.upload()
long_file = list(uploaded.keys())[0]

print("Upload boundary_corrected.xlsx")
uploaded = files.upload()
boundary_file = list(uploaded.keys())[0]

print("Upload original video")
uploaded = files.upload()
video_file = list(uploaded.keys())[0]

print("Upload original audio")
uploaded = files.upload()
orig_audio = list(uploaded.keys())[0]

print("Upload translated audio")
uploaded = files.upload()
trans_audio = list(uploaded.keys())[0]


# durations
y1, sr1 = librosa.load(orig_audio, sr=None)
y2, sr2 = librosa.load(trans_audio, sr=None)

orig_duration = len(y1)/sr1
trans_duration = len(y2)/sr2

scale = trans_duration / orig_duration

print("Original duration:",orig_duration)
print("Translated duration:",trans_duration)
print("Scale:",scale)


# if scale ~ 1 just replace audio
if abs(scale-1) < 0.02:

    print("Durations equal → no stretch needed")

    subprocess.run([
        "ffmpeg",
        "-i",video_file,
        "-i",trans_audio,
        "-map","0:v",
        "-map","1:a",
        "-c:v","copy",
        "-shortest",
        "final_lipsync_video.mp4"
    ])

else:

    print("Stretching video")

    subprocess.run([
        "ffmpeg",
        "-i",video_file,
        "-filter:v",f"setpts={1/scale}*PTS",
        "-an",
        "video_stretched.mp4"
    ])

    subprocess.run([
        "ffmpeg",
        "-i","video_stretched.mp4",
        "-i",trans_audio,
        "-map","0:v",
        "-map","1:a",
        "-shortest",
        "final_lipsync_video.mp4"
    ])


files.download("final_lipsync_video.mp4")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Upload long_silence.xlsx


Saving long_silence (4).xlsx to long_silence (4).xlsx
Upload boundary_corrected.xlsx


Saving boundary_corrected (1).xlsx to boundary_corrected (1).xlsx
Upload original video


Saving Untitled design (1).mp4 to Untitled design (1).mp4
Upload original audio


Saving english.wav to english.wav
Upload translated audio


Saving tamil.mp3 to tamil.mp3
Original duration: 8.89325
Translated duration: 10.410666666666666
Scale: 1.1706256617846869
Stretching video


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>